In [1]:
import os
import transformers
from helper_functions import *
import json





batch_path = "eval_p3"
transform_lct ="/work/eauten2s/ec_criteria_struct/lct"

# Model
model_id =  "meta-llama/Meta-Llama-3-70B-Instruct"
model_name = "Llama-3-70B-Instruct"

# N Shots
n_shot = 5 # liefert genau die Anzahl Beispiele (study, label)

# Input/ Output
study_path = f"{transform_lct}/input/lct_txt_half/"

# Model
model_id =  "meta-llama/Meta-Llama-3-70B-Instruct"
model_name = "Llama-3-70B-Instruct"
n_shot=2
# Input/ Output
study_path = f"{transform_lct}/input/lct_txt_half/"
output_path = f"{transform_lct}/evaluate/{batch_path}/model_output/{model_name}_{n_shot}_shot/output/"
os.makedirs(output_path, exist_ok=True)


anfang = 0
ende = 300
study_files = os.listdir(study_path)[anfang:ende]


# Load Model Description
model_desc = read_text_file(f"{transform_lct}/input/prompt/prompt_p3.txt")

def read_matching_p3_files(study_folder, n_shot):
    study_filenames = []
    study_contents = []
    label_filenames = []
    label_contents = []

    loaded_files = 0
    for file_name in os.listdir(study_folder):
        if file_name.endswith(".txt"):
            study_filenames.append(file_name)
            study_file_path = os.path.join(study_folder, file_name)
            with open(study_file_path, 'r', encoding='utf-8') as file:
                study_contents.append(file.read())

            label_file_name = file_name.replace(".txt", "_p3.json")
            label_file_path = os.path.join(study_folder, label_file_name)
            print(label_file_path)
            if os.path.exists(label_file_path):
                label_filenames.append(label_file_name)
                with open(label_file_path, 'r', encoding='utf-8') as file:
                    label_contents.append(file.read())
            else:
                label_filenames.append(None)
                label_contents.append(None)
            loaded_files += 1
            if loaded_files == n_shot * 2:
                break

    return study_filenames, study_contents, label_filenames, label_contents


# Load n-shot Data
study_folder = f"{transform_lct}/input/n_shot_files_p3"

study_filenames, study_contents, label_filenames, label_contents = read_matching_p3_files(study_folder, n_shot)
studies = dict(zip(study_filenames, study_contents))
labels = dict(zip(label_filenames, label_contents))
messages = []

command = "Bring the following eligibility criterias in Json format with logical operators and extract entitys:"

messages.append({"role": "system", "content": f"{model_desc}"})

for i in range(n_shot):
    messages.append({"role": "user", "content": f"{command} {studies[study_filenames[i]]}"})
    messages.append({"role": "assistant", "content": labels[label_filenames[i]]})





first_call = True

for file in study_files:

    print(file)
    test_file = read_text_file(study_path+file)
    print(test_file)

    if first_call:
        messages.append({"role": "user", "content": f"{command} {test_file}"})
        first_call = False
    else:
        messages[-1] = {"role": "user", "content": f"{command} {test_file}"}




/home/eauten2s/.local/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/work/eauten2s/ec_criteria_struct/lct/input/n_shot_files_p3/NCT03861819_inc_p3.json
/work/eauten2s/ec_criteria_struct/lct/input/n_shot_files_p3/NCT03865589_inc_p3.json
/work/eauten2s/ec_criteria_struct/lct/input/n_shot_files_p3/NCT03866200_exc_p3.json
/work/eauten2s/ec_criteria_struct/lct/input/n_shot_files_p3/NCT03868475_exc_p3.json
NCT03860012_exc.txt
Das File: /work/eauten2s/ec_criteria_struct/lct/input/lct_txt_half/NCT03860012_exc.txt wurde erfolgreich geladen.
1. abnormal folate levels
2. age > 21 or less than 2
NCT03860012_inc.txt
Das File: /work/eauten2s/ec_criteria_struct/lct/input/lct_txt_half/NCT03860012_inc.txt wurde erfolgreich geladen.
1. inflammatory bowel disease
2. on methotrexate at appropriate dosing
3. normal folate levels at onset of study
4. treatment with folic acid
5. ages 2-21 years
NCT03860025_exc.txt
Das File: /work/eauten2s/ec_criteria_struct/lct/input/lct_txt_half/NCT03860025_exc.txt wurde erfolgreich geladen.
-
NCT03860025_inc.txt
Das File: /work/eauten2s/e

In [2]:
messages

[{'role': 'system', 'content': 'Die Datei wurde nicht gefunden.'},
 {'role': 'user',
  'content': 'Bring the following eligibility criterias in Json format with logical operators and extract entitys: -  Ages 18-59\n-  Histologically confirmed nonalcoholic steatohepatitis and fibrosis (stage 1-3) from liver biopsy\n-  Able to row on an indoor rowing apparatus\n-  Not participating in regular physical exercise (more than 60 minutes of moderate intensity or 30 minutes of vigorous intensity exercise per week)\n-  Not actively trying to lose weight through a dietary approach\n-  No absolute contra-indications to exercise: recent (<6 months) acute cardiac event unstable angina, uncontrolled dysrhythmias causing symptoms or hemodynamic compromise, symptomatic aortic stenosis, uncontrolled symptomatic heart failure, acute pulmonary embolus, acute myocarditis or pericarditis, suspected or known dissecting aneurism and acute systemic infection\n-  No other form of chronic liver disease (as confi

### Test n-shot Input

In [3]:

import os
import transformers
from helper_functions import *

batch_path = "eval_p1"
n_prompt = 7
n_shot = 5
temp=0.6
temp_str = f"_temp_{str(temp).split('.')[1]}"
model_id =  "meta-llama/Meta-Llama-3-70B-Instruct"
model_name = "Llama-3-70B-Instruct"
transform_lct ="/work/eauten2s/ec_criteria_struct/lct"
model_desc = read_text_file(f"{transform_lct}/input/prompt/p{n_prompt}.txt")

study_path = f"{transform_lct}/input/lct_txt/"
output_path = f"{transform_lct}/evaluate/{batch_path}/model_output/{model_name}_{n_shot}_shot_prompt_{n_prompt}{temp_str}_cot/output/"
os.makedirs(output_path, exist_ok=True)


study_files = os.listdir(study_path)[:100]

shot_list = [
    "NCT03865433.txt",
    "NCT03860324.txt",
    "NCT03860233.txt",
    "NCT03923231.txt",
    "NCT03930121.txt"
]


# Load n-shot Data
study_folder = f"{transform_lct}/input/lct_txt/"
label_folder = f'{transform_lct}/input/lct_p1'
study_filenames, study_contents, label_filenames, label_contents = read_matching_txt_files(study_folder, label_folder, shot_list)
studies = dict(zip(study_filenames, study_contents))
labels = dict(zip(label_filenames, label_contents))
messages = []


cot = "Let's think through this carefully, step by step."

command = "Insert the logical operators [AND], [OR], [NOT] into the following eligibility criteria and return the text in full without deleting/replacing anything. Do not say anything else." 

messages.append({"role": "system", "content": f"{model_desc}"})

for i in range(n_shot):
    messages.append({"role": "user", "content": f"{command} {studies[study_filenames[i]]}"})
    messages.append({"role": "assistant", "content": labels[label_filenames[i]]})


first_call = True

for file in study_files:
    file_name = file.split(".")[0]
    print("File:", file_name, "\n")
    
    test_file = read_text_file(study_path+file)

    if first_call:
        messages.append({"role": "user", "content": f"{command} {test_file}"})
        first_call = False
    else:
        messages[-1] = {"role": "user", "content": f"{command} {test_file}"}


Das File: /work/eauten2s/ec_criteria_struct/lct/input/prompt/p7.txt wurde erfolgreich geladen.
Files NCT03865433.txt loaded.
Study: Inclusion Criteria:
  1. are aged 13-17, years and a high school student-athlete who attends school where an athletic trainer can supervise the exercise protocol;
  2. have sustained a concussion within 2-7 days of clinic presentation and diagnosed by a study physician;
  3. demonstrate symptom exacerbation during a graded treadmill exercise test and cannot complete the test;
  4. are currently participating in a school or club sport;
  5. are English speaking and capable of giving assent
Exclusion Criteria:
  1. have a reported history of neurological condition or disorder including but not limited to brain surgery, special education, seizure disorder, speech pathology, previous diagnosis of Post-Concussion Syndrome (PCS),
  2. are unwilling to exercise,
  3. have focal neurologic deficit that would represent risk for walking/running on treadmill,
  4. ex

### Test Random n-shot

In [4]:
import os
import random

def read_random_matching_txt_files(study_folder, label_folder, n):
    study_filenames = []
    label_filenames = []
    study_contents = []
    label_contents = []

    # Get all the filenames in the study folder
    all_filenames = [f for f in os.listdir(study_folder) if f.endswith('.txt')]

    # Randomly select n filenames
    selected_filenames = random.sample(all_filenames, n)

    for filename in selected_filenames:
        study_filepath = os.path.join(study_folder, filename)
        label_filepath = os.path.join(label_folder, filename)

        if os.path.isfile(study_filepath) and os.path.isfile(label_filepath):
            study_filenames.append(f"{filename}_study")
            label_filenames.append(f"{filename}_label")

            study_contents.append(read_file_content(study_filepath))
            label_contents.append(read_file_content(label_filepath))

    return study_filenames, study_contents, label_filenames, label_contents

In [5]:
# Load n-shot Data
study_folder = f"{transform_lct}/input/lct_txt/"
label_folder = f'{transform_lct}/input/lct_p1'
study_filenames, study_contents, label_filenames, label_contents = read_random_matching_txt_files(study_folder, label_folder, n_shot)
studies = dict(zip(study_filenames, study_contents))
labels = dict(zip(label_filenames, label_contents))
messages = []


cot = "Let's think through this carefully, step by step."

command = "Insert the logical operators [AND], [OR], [NOT] into the following eligibility criteria and return the text in full without deleting/replacing anything. Do not say anything else."

messages.append({"role": "system", "content": f"{model_desc}"})

for i in range(n_shot):
    messages.append({"role": "user", "content": f"{command} {studies[study_filenames[i]]}"})
    messages.append({"role": "assistant", "content": labels[label_filenames[i]]})


first_call = True

for file in study_files:
    file_name = file.split(".")[0]
    print("File:", file_name, "\n")

    test_file = read_text_file(study_path+file)

    if first_call:
        messages.append({"role": "user", "content": f"{command} {test_file}"})
        first_call = False
    else:
        messages[-1] = {"role": "user", "content": f"{command} {test_file}"}


File: NCT03860012 

Das File: /work/eauten2s/ec_criteria_struct/lct/input/lct_txt/NCT03860012.txt wurde erfolgreich geladen.
File: NCT03860025 

Das File: /work/eauten2s/ec_criteria_struct/lct/input/lct_txt/NCT03860025.txt wurde erfolgreich geladen.
File: NCT03860038 

Das File: /work/eauten2s/ec_criteria_struct/lct/input/lct_txt/NCT03860038.txt wurde erfolgreich geladen.
File: NCT03860064 

Das File: /work/eauten2s/ec_criteria_struct/lct/input/lct_txt/NCT03860064.txt wurde erfolgreich geladen.
File: NCT03860090 

Das File: /work/eauten2s/ec_criteria_struct/lct/input/lct_txt/NCT03860090.txt wurde erfolgreich geladen.
File: NCT03860103 

Das File: /work/eauten2s/ec_criteria_struct/lct/input/lct_txt/NCT03860103.txt wurde erfolgreich geladen.
File: NCT03860116 

Das File: /work/eauten2s/ec_criteria_struct/lct/input/lct_txt/NCT03860116.txt wurde erfolgreich geladen.
File: NCT03860142 

Das File: /work/eauten2s/ec_criteria_struct/lct/input/lct_txt/NCT03860142.txt wurde erfolgreich geladen.


In [6]:
messages

[{'role': 'system',
  'content': 'Welcome to the Medical Criteria Editing Challenge!\nYour mission is to skillfully incorporate the logical operators [AND], [OR], [NOT] into the provided medical inclusion and exclusion criteria.\nThis challenge will test your ability to enhance the clarity and specificity of the criteria while maintaining their semantic integrity.\n\nObjective:\nEdit the given criteria text by strategically inserting the logical operators [AND], [OR], [NOT] to maximize your score.\nAim to achieve a score that doubles the benchmark to ensure comprehensive coverage of all possibilities.\n\nGame Rules:\nPlacement Guidelines:\nInsert an [AND] before words like "with" to link related conditions or requirements.\nPlace an [OR] before "or", "and/or", after a comma.\nUse a [NOT] before negations like "no", "not", "unable".\n\nScoring System:\n\n- Based on an analysis of 100 input files, the average counts are: 85 [AND], 486 [OR], 106 [NOT].\n- Your goal is to double these numb

In [7]:
study_filenames

['NCT03863548.txt_study',
 'NCT03867409.txt_study',
 'NCT03869021.txt_study',
 'NCT03921736.txt_study',
 'NCT03920514.txt_study']

In [8]:
label_filenames

['NCT03863548.txt_label',
 'NCT03867409.txt_label',
 'NCT03869021.txt_label',
 'NCT03921736.txt_label',
 'NCT03920514.txt_label']

In [9]:
label_contents

['Inclusion Criteria:\n - person who has expressed willingness to participate\n - person over 18 years of age\n - person with a retinal-vitreous condition requiring scheduled surgery (epi-retinal membrane surgery, [OR] vitreous traction surgery, [OR] macular hole surgery).\nExclusion Criteria:\n - person subject to legal protection (curatorship, guardianship)\n - person deemed mentally incompetent\n - pregnant, [OR] parturient [OR] or breastfeeding woman\n - adult unwilling or unable to consent\n - patient who has already participated in the study\n - person with a physical [OR] or mental disability that does [NOT] not allow participation.\n - a person who has participated in any study of an experimental medical product within the previous 3 months\n - person who experiences any of the following during the ophthalmological examination:\n - severe [OR] or proliferating diabetic retinopathy\n - intra-vitreal hemorrhage\n - tractional retinal detachment\n - subretinal [OR] or retrohyaloid

In [10]:
study_contents

['Inclusion Criteria:\n  -  person who has expressed willingness to participate\n  -  person over 18 years of age\n  -  person with a retinal-vitreous condition requiring scheduled surgery (epi-retinal membrane surgery, vitreous traction surgery, macular hole surgery).\nExclusion Criteria:\n  -  person subject to legal protection (curatorship, guardianship)\n  -  person deemed mentally incompetent\n  -  pregnant, parturient or breastfeeding woman\n  -  adult unwilling or unable to consent\n  -  patient who has already participated in the study\n  -  person with a physical or mental disability that does not allow participation.\n  -  a person who has participated in any study of an experimental medical product within the previous 3 months\n  -  person who experiences any of the following during the ophthalmological examination:\n       -  severe or proliferating diabetic retinopathy\n       -  intra-vitreal hemorrhage\n       -  tractional retinal detachment\n       -  subretinal or ret

### Füge Text ein in Prompt

In [11]:
model_desc = read_text_file(f"{transform_lct}/input/prompt/p{10}.txt")

Das File: /work/eauten2s/ec_criteria_struct/lct/input/prompt/p10.txt wurde erfolgreich geladen.


In [12]:
model_desc.replace("<criteria>\n{$CRITERIA_TEXT}\n</criteria>", test_file)

'Here is some inclusion and exclusion criteria text:\n\nInclusion Criteria:\n  -  Patients who are day -10 pre- to day +30 post-allogeneic hematopoietic cell transplant (AHCT) from any donor or graft source and for any conditioning regimen\n  -  Patients who have received treatment with meropenem or piperacillin-tazobactam (pip-tazo) intravenously (IV) (of at least 24 hours duration) in past 7 days\n  -  Controlled infection defined as hemodynamically stable and not requiring supplemental oxygen of more than 2 liters via nasal cannula\n  -  Patients who are able to take oral medications in suspension form\n  -  Patients who are able to provide informed consent (IC) and comply with all study visits and procedures\nExclusion Criteria:\n  -  Patients who are anticipated to require continued broad spectrum antibiotics with meropenem or pip-tazo IV for > 96 hours post-engraftment such as for known, documented infections necessitating prolonged treatment\n  -  Patients with a prior documente

In [13]:
studies

{'NCT03863548.txt_study': 'Inclusion Criteria:\n  -  person who has expressed willingness to participate\n  -  person over 18 years of age\n  -  person with a retinal-vitreous condition requiring scheduled surgery (epi-retinal membrane surgery, vitreous traction surgery, macular hole surgery).\nExclusion Criteria:\n  -  person subject to legal protection (curatorship, guardianship)\n  -  person deemed mentally incompetent\n  -  pregnant, parturient or breastfeeding woman\n  -  adult unwilling or unable to consent\n  -  patient who has already participated in the study\n  -  person with a physical or mental disability that does not allow participation.\n  -  a person who has participated in any study of an experimental medical product within the previous 3 months\n  -  person who experiences any of the following during the ophthalmological examination:\n       -  severe or proliferating diabetic retinopathy\n       -  intra-vitreal hemorrhage\n       -  tractional retinal detachment\n  

In [14]:
messages = [
        {
            "role": "user", 
            "content": [
                {
                    "type": "text",
                    "text": f"Example {i+1}:\nStudy:\n{studies[shot]}\nLabeled Criteria:\n{labels[shot.replace('.txt', '_p1.txt')]}\n\n"
                } for i, shot in enumerate(shot_list)
                if shot in studies and shot.replace('.txt', '_p1.txt') in labels
            ] + [
                {
                    "type": "text",
                    "text": f"Test File:\n{test_file}"
                }
            ]
        }
    ]

In [18]:
additional_files = [
    "NCT03865433.txt",
    "NCT03860324.txt",
    "NCT03860233.txt",
    "NCT03923231.txt",
    "NCT03930121.txt",
    "NCT03863717.txt",
    "NCT03863925.txt",
    "NCT03863951.txt",
    "NCT03865134.txt",
    "NCT03868267.txt",
    "NCT03929640.txt",
    "NCT03861845.txt"
]

# Load n-shot Data
study_folder = f"{transform_lct}/input/lct_txt/"
label_folder = f'{transform_lct}/input/lct_p1'
study_filenames, study_contents, label_filenames, label_contents = read_matching_txt_files(study_folder, label_folder, additional_files)


studies = dict(zip(study_filenames, study_contents))
labels = dict(zip(label_filenames, label_contents))
messages = []

cot = "Let's think through this carefully, step by step."

command = read_text_file(f"{transform_lct}/input/prompt/command.txt")
messages.append({"role": "system", "content": f"{model_desc}"})
n_shot=12
for i in range(n_shot):
    messages.append({"role": "user", "content": f"{command} {studies[study_filenames[i]]}"})
    messages.append({"role": "assistant", "content": labels[label_filenames[i]]})

Files NCT03865433.txt loaded.
Study: Inclusion Criteria:
  1. are aged 13-17, years and a high school student-athlete who attends school where an athletic trainer can supervise the exercise protocol;
  2. have sustained a concussion within 2-7 days of clinic presentation and diagnosed by a study physician;
  3. demonstrate symptom exacerbation during a graded treadmill exercise test and cannot complete the test;
  4. are currently participating in a school or club sport;
  5. are English speaking and capable of giving assent
Exclusion Criteria:
  1. have a reported history of neurological condition or disorder including but not limited to brain surgery, special education, seizure disorder, speech pathology, previous diagnosis of Post-Concussion Syndrome (PCS),
  2. are unwilling to exercise,
  3. have focal neurologic deficit that would represent risk for walking/running on treadmill,
  4. exhibit an inability to exercise due to injury, known heart disease, or increased cardiac risk,
 

In [16]:
messages

[{'role': 'system',
  'content': 'Here is some inclusion and exclusion criteria text:\n\n<criteria>\n{$CRITERIA_TEXT}\n</criteria>\n\nYour task is to edit this text to incorporate the logical operators [AND], [OR], and [NOT] according\nto the following rules:\n\n[OR]:\n- Only annotate [OR] between criteria explicitly separated by "or" or "and/or".\n- Avoid clutter by not annotating commas as [OR].\n\n[AND]:\n- Use [AND] to indicate a conjunction between two events or entities.\n- Include [AND] before words like "and" or "with" when they connect related conditions or\nrequirements that must all be satisfied.\n- Do not annotate [AND] when used in numeric comparisons (e.g., age ranges).\n\n[NOT]:\n- Always write a [NOT] before "no", "not", "unable", "none", "inadequate", "unqualified",\n"Inability", or "doesn\'t".\n\nAdditional Guidelines:\n- Ensure that [AND], [OR], or [NOT] do not appear at the beginning of a line or immediately after a\nline break.\n- Aim for clarity and only add opera